# Supply Chain Control Tower — cleaning & EDA

Sanity checks on the star-schema parquet before the Power BI control tower.
Focus: revenue flags, OTIF / perfect-order rates, freight & CO2, inventory on-hand.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("../data")
OUT = Path("../python/outputs")
OUT.mkdir(parents=True, exist_ok=True)

orders = pd.read_parquet(DATA / "fact_orders.parquet")
ship = pd.read_parquet(DATA / "fact_shipments.parquet")
inv = pd.read_parquet(DATA / "fact_inventory.parquet")
proc = pd.read_parquet(DATA / "fact_procurement.parquet")
tm = pd.read_parquet(DATA / "dim_transport_mode.parquet")

print("orders", orders.shape, "ship", ship.shape, "inv", inv.shape, "po", proc.shape)
orders.head(2)


In [ ]:
# flag mix + unknown keys
print(orders[["is_revenue","is_canceled","is_fraud","is_otif","is_perfect_order"]].mean().round(3))
print("vendor_key < 0:", (orders["vendor_key"] < 0).sum())
print("dup order_line_key:", orders["order_line_key"].duplicated().sum())


In [ ]:
# executive KPIs (revenue lines)
rev = orders[orders["is_revenue"] == True]
if rev.empty:
    rev = orders[orders["is_revenue"] == 1]

o = orders.groupby("order_id").agg(otif=("is_otif","min"), perfect=("is_perfect_order","min"))
print(f"revenue ${rev['net_sales'].sum()/1e6:.1f}M")
print(f"profit  ${rev['line_profit'].sum()/1e6:.1f}M")
print(f"orders  {orders['order_id'].nunique():,}")
print(f"AOV     ${rev.groupby('order_id')['net_sales'].sum().mean():.0f}")
print(f"OTIF    {100*o['otif'].mean():.1f}%")
print(f"perfect {100*o['perfect'].mean():.1f}%")


In [ ]:
# logistics
print(f"freight ${ship['freight_cost'].sum()/1e6:.1f}M")
print(f"late %  {100*ship['is_late'].mean():.1f}")
print(f"CO2 t   {ship['co2_kg'].sum()/1000:.1f}")

mode = ship.merge(tm, on="transport_mode_key")
mix = mode.groupby("mode_name").size().sort_values(ascending=False)
print(mix)
print((100 * mix / mix.sum()).round(1))

fig, ax = plt.subplots(figsize=(6, 3.5))
mix.plot(kind="bar", ax=ax, color="#0D47A1")
ax.set_title("Shipments by transport mode")
ax.set_ylabel("Shipments")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
fig.savefig(OUT / "shipments_by_mode.png", dpi=120)
plt.show()


In [ ]:
# inventory on-hand trend (daily total value)
daily_inv = inv.groupby("date_key")["on_hand_value"].sum().reset_index()
daily_inv["date"] = pd.to_datetime(daily_inv["date_key"].astype(str), format="%Y%m%d")

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(daily_inv["date"], daily_inv["on_hand_value"] / 1e6, color="#0D47A1")
ax.set_ylabel("On-hand $M")
ax.set_title("Inventory on-hand value")
plt.xticks(rotation=45)
plt.tight_layout()
fig.savefig(OUT / "inventory_onhand.png", dpi=120)
plt.show()
print("latest on-hand $M", round(daily_inv["on_hand_value"].iloc[-1] / 1e6, 1))


In [ ]:
# procurement quick look
print("PO spend $M", round(proc["po_value"].sum() / 1e6, 1))
print("PO lines", len(proc))
print("avg SLA days", proc["sla_days"].mean())
print("on-time receipt %", round(100 * proc["is_on_time_receipt"].mean(), 1))
